# Day 56 · Exercise 3: Safe Storage

**What you'll build:** Implement `safe_filename(original)` and `save_upload(content, filename, upload_dir)`. Safe filenames prevent path traversal attacks; a UUID prefix prevents two users uploading `report.pdf` from overwriting each other.

## Setup (provided)

In [ ]:
import re
import secrets
import tempfile
from pathlib import Path


## Your Implementation

In [ ]:
def safe_filename(original: str) -> str:
    """Return a filesystem-safe filename with a random prefix.

    Args:
        original: The original filename (may contain spaces, path separators, etc.)
    Returns:
        A safe filename: random_prefix + sanitized_basename.
        - Uses Path(original).name to strip directory components.
        - Replaces any character that is not \w, hyphen, or dot with underscore.
        - Prepends secrets.token_hex(4) to avoid collisions.
    """
    # TODO: name = Path(original).name
    # name = re.sub(r'[^\w\-.]', '_', name)
    # return f"{secrets.token_hex(4)}_{name}"
    raise NotImplementedError

def save_upload(content: bytes, filename: str, upload_dir: Path) -> Path:
    """Write bytes to upload_dir/safe_filename(filename) and return the path.

    Args:
        content:    Raw bytes to write.
        filename:   Original filename (will be made safe).
        upload_dir: Directory to write into (created if missing).
    Returns:
        The Path where the file was written.
    """
    # TODO: upload_dir.mkdir(parents=True, exist_ok=True)
    # dest = upload_dir / safe_filename(filename)
    # dest.write_bytes(content)
    # return dest
    raise NotImplementedError


In [ ]:
def safe_filename(original: str) -> str:
    name = Path(original).name
    name = re.sub(r'[^\w\-.]', '_', name)
    return f"{secrets.token_hex(4)}_{name}"

def save_upload(content: bytes, filename: str, upload_dir: Path) -> Path:
    upload_dir.mkdir(parents=True, exist_ok=True)
    dest = upload_dir / safe_filename(filename)
    dest.write_bytes(content)
    return dest


## Check Your Work

In [ ]:
def _run_checks():
    score = 0
    total = 5

    def _chk(n, ok, msg):
        nonlocal score
        print(f"  {'✅' if ok else '❌'} Check {n}: {msg}")
        if ok:
            score += 1

    try:
        name = safe_filename("hello world.txt")
    except NotImplementedError:
        for i in range(1, total + 1):
            print(f"  ❌ Check {i}: safe_filename not implemented")
        print(f"\nScore: 0 / {total}")
        return

    _chk(1, isinstance(name, str) and name.endswith(".txt"),
         f"safe_filename preserves extension (got {name!r})")
    _chk(2, " " not in name,
         f"spaces replaced (got {name!r})")

    path_input = "../../etc/passwd"
    safe = safe_filename(path_input)
    _chk(3, "/" not in safe and "\\" not in safe,
         f"path traversal stripped (got {safe!r})")

    n1 = safe_filename("file.txt")
    n2 = safe_filename("file.txt")
    _chk(4, n1 != n2, "two calls on same name → different results (random prefix)")

    try:
        import tempfile
        with tempfile.TemporaryDirectory() as td:
            dest = save_upload(b"test content", "my file.txt", Path(td))
            _chk(5, dest.exists() and dest.read_bytes() == b"test content",
                 f"save_upload writes correct bytes to {dest.name!r}")
    except NotImplementedError:
        print(f"  ❌ Check 5: save_upload not implemented")

    print(f"\nScore: {score} / {total}")
    if score == total:
        print("🎉 Exercise complete!")

_run_checks()


## Bonus Challenge

Extend `save_upload` to return a dict `{'path': str(dest), 'size': dest.stat().st_size}` instead of just the Path. Add a check that `size` matches `len(content)`. In production, you'd also compute a SHA-256 checksum of the content here to verify file integrity later.

## Solution

<details>
<summary>Show solution</summary>

```python
def safe_filename(original: str) -> str:
    name = Path(original).name
    name = re.sub(r'[^\w\-.]', '_', name)
    return f"{secrets.token_hex(4)}_{name}"

def save_upload(content: bytes, filename: str, upload_dir: Path) -> Path:
    upload_dir.mkdir(parents=True, exist_ok=True)
    dest = upload_dir / safe_filename(filename)
    dest.write_bytes(content)
    return dest
```

**Why this works:** `Path(original).name` strips any directory prefix — so
`../../etc/passwd` becomes just `passwd`, preventing path traversal. The regex
replaces anything that isn't a word character, hyphen, or dot with an underscore,
so spaces and special chars become safe. Prepending `secrets.token_hex(4)` gives
8 hex characters of randomness, making collisions (two users upload `report.txt`)
essentially impossible. `Path.write_bytes` creates or overwrites the file atomically.

</details>